# Demo
Training an **MLP**, **CNN**, **LSTM** and **TinyVIT** using our implementation vs JAX vs Opacus.

In [1]:
import os
os.chdir("/root/torch_jit")

In [2]:
import math
import time
import itertools
from dataclasses import dataclass
from typing import Optional

import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
import torchvision.transforms as transforms
from torch.utils.data import DataLoader, Subset, TensorDataset
import jax
import jax.numpy as jnp
from opacus import PrivacyEngine


from custom_models import ModelSpec, build_model
from opacus_models import build_opacus_model
from jax_models import _jax_model_factory, train_jax_dp

from torch_privacy import CustomPrivacyEngine, DPConfig



In [3]:
torch.cuda.is_available()

True

In [4]:
device = "cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu"

def get_mnist_loader(batch_size: int, train: bool = True) -> DataLoader:
    transform = transforms.Compose([transforms.ToTensor()])
    dataset = torchvision.datasets.MNIST(
        root="/root/torch-jit/data", train=train, download=True, transform=transform
    )
    return torch.utils.data.DataLoader(
        dataset, batch_size=batch_size, shuffle=train, drop_last=True
    )

def get_synthetic_sequence_loader(
    batch_size: int,
    seq_len: int = 64,
    input_dim: int = 64,
    num_classes: int = 10,
    size: int = 4096,
    train: bool = True,
 ) -> DataLoader:
    """
    Generates a synthetic sequence dataset and wrap it in a DataLoader.
    """
    g = torch.Generator().manual_seed(123 if train else 321)
    x = torch.randn(size, seq_len, input_dim, generator=g)

    # Correlate labels with signal in first few features for learnability.
    logits = x[:, :, : min(input_dim, num_classes)].mean(dim=1)
    y = logits.argmax(dim=1) % num_classes

    ds = TensorDataset(x, y)
    return DataLoader(ds, batch_size=batch_size, shuffle=train)

In [5]:
custom_mlp = build_model(ModelSpec("mlp", num_classes=10, in_channels=64, image_size=28))
opacus_mlp = build_opacus_model(ModelSpec("mlp", num_classes=10, in_channels=64, image_size=28))
jax_mlp = _jax_model_factory("mlp", ModelSpec("mlp", num_classes=10, in_channels=64, image_size=28), key=jax.random.PRNGKey(0))

In [ ]:
print(custom_mlp)
print(opacus_mlp)
print(jax_mlp)

MLP(
  (fc1): Linear(in_features=50176, out_features=256, bias=True)
  (fc2): Linear(in_features=256, out_features=128, bias=True)
  (fc3): Linear(in_features=128, out_features=10, bias=True)
)
OpacusMLP(
  (fc1): Linear(in_features=50176, out_features=256, bias=True)
  (fc2): Linear(in_features=256, out_features=128, bias=True)
  (fc3): Linear(in_features=128, out_features=10, bias=True)
)
({'linear': {'w': Array([[-0.00841537, -0.00767982,  0.0008753 , ..., -0.00168549,
         0.00138097,  0.0035038 ],
       [ 0.00047527,  0.00338183,  0.00223741, ...,  0.00456225,
        -0.0007031 , -0.00062131],
       [-0.00022659,  0.00761877,  0.00306787, ...,  0.00115348,
        -0.00494244, -0.00589231],
       ...,
       [-0.0037557 ,  0.00171832, -0.00511648, ..., -0.00410257,
        -0.00873256, -0.00062754],
       [-0.00860939, -0.00716944, -0.00572468, ..., -0.00729834,
        -0.00494411,  0.00033764],
       [-0.00364071,  0.00754952,  0.0020761 , ...,  0.00183718,
        -0.

In [6]:
def _sync_device(device):
    if device == "cuda" and torch.cuda.is_available():
        torch.cuda.synchronize()
    elif device == "mps" and hasattr(torch, "mps") and torch.backends.mps.is_available():
        torch.mps.synchronize()

In [7]:
# Non-private training function
def train_non_private(
    model_name,
    spec,
    train_loader,
    test_loader,
    max_steps,
    warmup_steps,
    lr,
 ):
    model = build_model(spec)
    model.to(device)
    optimizer = torch.optim.SGD(model.parameters(), lr=lr)

    losses = []
    accs = []
    steps = 0

    stream = itertools.cycle(train_loader)
    for _ in range(warmup_steps):
        x, y = next(stream)
        x, y = x.to(device), y.to(device)
        optimizer.zero_grad()
        logits = model(x)
        loss = F.cross_entropy(logits, y)
        loss.backward()
        optimizer.step()

    _sync_device(device)
    start = time.perf_counter()
    for _ in range(max_steps):
        x, y = next(stream)
        x, y = x.to(device), y.to(device)
        optimizer.zero_grad()
        logits = model(x)
        loss = F.cross_entropy(logits, y)
        loss.backward()
        optimizer.step()
        losses.append(loss.item())
        accs.append((logits.argmax(dim=-1) == y).float().mean().item())
        steps += 1
        if steps >= max_steps:
            break
    _sync_device(device)
    elapsed = time.perf_counter() - start

    test_accs = []
    with torch.no_grad():
        for x, y in test_loader:
            x, y = x.to(device), y.to(device)
            logits = model(x)
            test_accs.append((logits.argmax(dim=-1) == y).float().mean().item())

    return {
        "train_loss": sum(losses) / max(len(losses), 1),
        "train_acc": sum(accs) / max(len(accs), 1),
        "test_acc": sum(test_accs) / max(len(test_accs), 1),
        "seconds": elapsed,
        "steps": steps,
    }

# Custom Privacy Engine Training Function
def train_custom_dp(
    model_name,
    spec,
    dp_config,
    train_loader,
    test_loader,
    max_steps,
    warmup_steps,
    lr,
    max_grad_norm,
    noise_multiplier,
    use_compile=False
 ):
    model = build_model(spec)
    model.to(device)
    optimizer = torch.optim.SGD(model.parameters(), lr=lr)
    engine = CustomPrivacyEngine(
        model,
        config = dp_config
    )

    losses = []
    accs = []
    steps = 0

    stream = itertools.cycle(train_loader)
    for _ in range(warmup_steps):
        x, y = next(stream)
        x, y = x.to(device), y.to(device)
        stats = engine.dp_step(x, y, optimizer, loss_fn=F.cross_entropy, use_compile=use_compile)

    _sync_device(device)
    start = time.perf_counter()
    for _ in range(max_steps):
        x, y = next(stream)
        x, y = x.to(device), y.to(device)
        stats = engine.dp_step(x, y, optimizer, loss_fn=F.cross_entropy, use_compile=use_compile)
        losses.append(stats["loss"])
        accs.append(stats["acc"])
        steps += 1
        if steps >= max_steps:
            break
    _sync_device(device)
    elapsed = time.perf_counter() - start

    test_accs = []
    with torch.no_grad():
        for x, y in test_loader:
            x, y = x.to(device), y.to(device)
            logits = model(x)
            test_accs.append((logits.argmax(dim=-1) == y).float().mean().item())

    return {
        "train_loss": sum(losses) / max(len(losses), 1),
        "train_acc": sum(accs) / max(len(accs), 1),
        "test_acc": sum(test_accs) / max(len(test_accs), 1),
        "seconds": elapsed,
        "steps": steps,
    }

# Opacus Training Function
def train_opacus_dp(
    model_name,
    spec,
    train_loader,
    test_loader,
    max_steps,
    warmup_steps,
    lr,
    max_grad_norm,
    noise_multiplier,
    grad_sample_mode="hooks",
 ):
    model = build_opacus_model(spec)
    model.to(device)

    optimizer = torch.optim.SGD(model.parameters(), lr=lr)

    privacy_engine = PrivacyEngine()

    model, optimizer, train_loader = privacy_engine.make_private(
        module=model,
        optimizer=optimizer,
        data_loader=train_loader,
        noise_multiplier=noise_multiplier,
        max_grad_norm=max_grad_norm,
        grad_sample_mode=grad_sample_mode
    )
    criterion = nn.CrossEntropyLoss(reduction="mean")
    model.train()

    losses = []
    accs = []
    steps = 0

    stream = itertools.cycle(train_loader)
    for _ in range(warmup_steps):
        x, y = next(stream)
        x, y = x.to(device), y.to(device)
        optimizer.zero_grad()
        logits = model(x)
        loss = F.cross_entropy(logits, y)
        loss.backward()
        optimizer.step()

    start = time.perf_counter()
    for _ in range(max_steps):
        x, y = next(stream)
        x, y = x.to(device), y.to(device)
        optimizer.zero_grad()
        logits = model(x)
        loss = F.cross_entropy(logits, y)
        loss.backward()
        optimizer.step()
        losses.append(loss.item())
        accs.append((logits.argmax(dim=-1) == y).float().mean().item())
        steps += 1
        if steps >= max_steps:
            break

    elapsed = time.perf_counter() - start

    test_accs = []
    with torch.no_grad():
        for x, y in test_loader:
            x, y = x.to(device), y.to(device)
            logits = model(x)
            test_accs.append((logits.argmax(dim=-1) == y).float().mean().item())

    return {
        "train_loss": sum(losses) / max(len(losses), 1),
        "train_acc": sum(accs) / max(len(accs), 1),
        "test_acc": sum(test_accs) / max(len(test_accs), 1),
        "seconds": elapsed,
        "steps": steps,
    }

# Diagnostics

In [11]:
device = "cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu"



engine = CustomPrivacyEngine(
    model=build_model(ModelSpec(model_name="lstm", seq_input_dim=64, seq_len=32, seq_hidden_dim=128, num_classes=10)).to(device),
    config=DPConfig()
)
def explain_compile():
    lstm_model = build_model(ModelSpec(model_name="lstm", seq_input_dim=64, seq_len=32, seq_hidden_dim=128, num_classes=10)).to(device)


    if not hasattr(torch, "_dynamo"):
        print("torch._dynamo is not available")
        return
    # Build a tiny batch to trace
    loader = get_synthetic_sequence_loader(batch_size=4, seq_len=32, input_dim=64, num_classes=10, size=16, train=True)

    x, y = next(iter(loader))
    x = x.to(device)
    y = y.to(device)

    params = {k: v.detach() for k, v in lstm_model.named_parameters()}
    buffers = {k: v for k, v in lstm_model.named_buffers()}
    
    compiled_step = engine.get_compiled_step("flat", F.cross_entropy)

    report = torch._dynamo.explain(
        compiled_step,
        params,
        buffers,
        (x, y),
        float(engine.config.max_grad_norm),
        float(engine.config.noise_multiplier),
        0.1,  # lr
    )
    print(report)

explain_compile()

/root/miniconda3/envs/myenv/lib/python3.10/unittest/mock.py:1370: FutureWarning: explain(f, *args, **kwargs) is deprecated, use explain(f)(*args, **kwargs) instead.  If you don't migrate, we may break your explain call in the future if your user defined kwargs conflict with future kwargs added to explain(f).
  return func(*newargs, **newkeywargs)


Graph Count: 1
Graph Break Count: 0
Op Count: 623
Break Reasons:
Ops per Graph:
  Ops 1:
    <function lazy_load_decompositions at 0x760e1d1f2710>
    <built-in function getitem>
    <built-in function getitem>
    <function _vmap_increment_nesting at 0x760e1d1f25f0>
    <function _add_batch_dim at 0x760e1d1f24d0>
    <function _add_batch_dim at 0x760e1d1f24d0>
    <built-in method _saved_tensors_hooks_disable of pybind11_builtins.pybind11_detail_function_record_v1_system_libstdcpp_gxx_abi_1xxx_use_cxx11_abi_1 object at 0x760e1d7f7fb0>
    <built-in method _grad_increment_nesting of pybind11_builtins.pybind11_detail_function_record_v1_system_libstdcpp_gxx_abi_1xxx_use_cxx11_abi_1 object at 0x760f40db9e50>
    <built-in method _wrap_for_grad of pybind11_builtins.pybind11_detail_function_record_v1_system_libstdcpp_gxx_abi_1xxx_use_cxx11_abi_1 object at 0x760f40db9ed0>
    <built-in method _wrap_for_grad of pybind11_builtins.pybind11_detail_function_record_v1_system_libstdcpp_gxx_abi_1xxx

## Run a small benchmark
This runs a quick sanity check across the three DP implementations. Adjust `max_steps` or `models_to_run` for a longer benchmark.

In [8]:
from rich.console import Console
from rich.table import Table
import gc

batch_size = 128
max_steps = 1000
warmup_steps = 1
lr = 1e-3
max_grad_norm = 1.0
noise_multiplier = 1.1

dp_config = DPConfig(
    max_grad_norm=max_grad_norm,
    noise_multiplier=noise_multiplier,
    vmap_chunk_size=32,
    force_compile_mps=True,
 )

mnist_train = get_mnist_loader(batch_size, train=True)
mnist_test = get_mnist_loader(batch_size, train=False)

seq_train = get_synthetic_sequence_loader(batch_size, size=2048, train=True)
seq_test = get_synthetic_sequence_loader(batch_size, size=1024, train=False)

specs = {
    "mlp": ModelSpec("mlp", num_classes=10, in_channels=1, image_size=28),
    "cnn": ModelSpec("cnn", num_classes=10, in_channels=1, image_size=28),
    "lstm": ModelSpec("lstm", num_classes=10, seq_len=64, seq_input_dim=64, seq_hidden_dim=128),
    "vit": ModelSpec("vit", num_classes=10, in_channels=1, image_size=28),
}

loaders = {
    "mlp": (mnist_train, mnist_test),
    "cnn": (mnist_train, mnist_test),
    "vit": (mnist_train, mnist_test),
    "lstm": (seq_train, seq_test),
}

models_to_run = ["mlp", "cnn", "lstm"] 

def _cleanup_cuda():
    if torch.cuda.is_available():
        torch.cuda.synchronize()
        torch.cuda.empty_cache()
    gc.collect()

results = []
for model_name in models_to_run:
    spec = specs[model_name]
    train_loader, test_loader = loaders[model_name]
    # Non-private baseline
    non_private_stats = train_non_private(
        model_name,
        spec,
        train_loader,
        test_loader,
        max_steps=max_steps,
        warmup_steps=warmup_steps,
        lr=lr,
    )
    results.append(("non_private", model_name, non_private_stats))

    # # Eager Execution (Private)
    custom_stats_eager = train_custom_dp(
        model_name,
        spec,
        dp_config,
        train_loader,
        test_loader,
        max_steps=max_steps,
        warmup_steps=warmup_steps,
        lr=lr,
        max_grad_norm=max_grad_norm,
        noise_multiplier=noise_multiplier,
        use_compile=False,
    )
    results.append(("custom_eager", model_name, custom_stats_eager))

    # # Compiled Execution (Private)
    custom_stats_compiled = train_custom_dp(
        model_name,
        spec,
        dp_config,
        train_loader,
        test_loader,
        max_steps=max_steps,
        warmup_steps=warmup_steps,
        lr=lr,
        max_grad_norm=max_grad_norm,
        noise_multiplier=noise_multiplier,
        use_compile=True,
    )
    results.append(("custom_compiled", model_name, custom_stats_compiled))

    #JAX Execution (Private)
    jax_stats = train_jax_dp(
        model_name,
        spec,
        train_loader,
        test_loader,
        max_steps=max_steps,
        warmup_steps=warmup_steps,
        lr=lr,
        max_grad_norm=max_grad_norm,
        noise_multiplier=noise_multiplier,
        use_vmap=True,
        use_jit=True,
    )
    results.append(("jax", model_name, jax_stats))

    # # Opacus Execution (Private)
    opacus_stats = train_opacus_dp(
        model_name,
        spec,
        train_loader,
        test_loader,
        max_steps=max_steps,
        warmup_steps=warmup_steps,
        lr=lr,
        max_grad_norm=max_grad_norm,
        noise_multiplier=noise_multiplier,
        grad_sample_mode="hooks",
    )
    results.append(("opacus", model_name, opacus_stats))

    del custom_stats_eager, custom_stats_compiled, jax_stats, opacus_stats
    _cleanup_cuda()



2026-04-20 09:04:41.184284: W external/xla/xla/service/gpu/autotuning/dot_search_space.cc:200] All configs were filtered out because none of them sufficiently match the hints. Maybe the hints set does not contain a good representative set of valid configs?Working around this by using the full hints set instead.
/root/miniconda3/envs/myenv/lib/python3.10/site-packages/opacus/privacy_engine.py:96: UserWarning: Secure RNG turned off. This is perfectly fine for experimentation as it allows for much faster training performance, but remember to turn it on and retrain one last time before production with ``secure_mode`` turned on.
  warnings.warn(
04/20/2026 09:04:57:WARNING:Ignoring drop_last as it is not compatible with DPDataLoader.
sys:1: UserWarning: Full backward hook is firing when gradients are computed with respect to module outputs since no inputs require gradients. See https://docs.pytorch.org/docs/main/generated/torch.nn.Module.html#torch.nn.Module.register_full_backward_hook for 

In [9]:
table = Table(title="DP Benchmark Results", show_lines=True)
table.add_column("Backend", style="bold cyan")
table.add_column("Model", style="bold magenta")
table.add_column("Train Loss", justify="right", style="green")
table.add_column("Train Acc", justify="right", style="green")
table.add_column("Test Acc", justify="right", style="bright_blue")
table.add_column("Seconds", justify="right", style="yellow")

for backend, model_name, stats in results:
    table.add_row(
        backend,
        model_name,
        f"{stats['train_loss']:.4f}",
        f"{stats['train_acc']:.4f}",
        f"{stats['test_acc']:.4f}",
        f"{stats['seconds']:.3f}",
    )

Console().print(table)

                          DP Benchmark Results                           
┏━━━━━━━━━━━━━━━━━┳━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━┳━━━━━━━━━━┳━━━━━━━━━┓
┃ Backend         ┃ Model ┃ Train Loss ┃ Train Acc ┃ Test Acc ┃ Seconds ┃
┡━━━━━━━━━━━━━━━━━╇━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━╇━━━━━━━━━━╇━━━━━━━━━┩
│ non_private     │ mlp   │     2.2862 │    0.1512 │   0.2309 │   7.118 │
├─────────────────┼───────┼────────────┼───────────┼──────────┼─────────┤
│ custom_eager    │ mlp   │     2.2992 │    0.1121 │   0.1253 │  21.073 │
├─────────────────┼───────┼────────────┼───────────┼──────────┼─────────┤
│ custom_compiled │ mlp   │     2.2948 │    0.1375 │   0.1500 │   6.703 │
├─────────────────┼───────┼────────────┼───────────┼──────────┼─────────┤
│ jax             │ mlp   │     2.2938 │    0.1283 │   0.1711 │   4.002 │
├─────────────────┼───────┼────────────┼───────────┼──────────┼─────────┤
│ opacus          │ mlp   │     2.2989 │    0.1252 │   0.1264 │  15.348 │
├─────────────────┼───────┼────────────┼───────────┼──────────┼─────────┤
│ non_private     │ cnn   │     2.2714 │    0.2769 │   0.5015 │   7.981 │
├─────────────────┼───────┼────────────┼───────────┼──────────┼─────────┤
│ custom_eager    │ cnn   │     2.2935 │    0.1271 │   0.1474 │  35.024 │
├─────────────────┼───────┼────────────┼───────────┼──────────┼─────────┤
│ custom_compiled │ cnn   │     2.2958 │    0.1031 │   0.1373 │  10.758 │
├─────────────────┼───────┼────────────┼───────────┼──────────┼─────────┤
│ jax             │ cnn   │     2.3123 │    0.0371 │   0.0571 │   5.344 │
├─────────────────┼───────┼────────────┼───────────┼──────────┼─────────┤
│ opacus          │ cnn   │     2.2969 │    0.1145 │   0.1761 │  18.828 │
├─────────────────┼───────┼────────────┼───────────┼──────────┼─────────┤
│ non_private     │ lstm  │     2.3027 │    0.1009 │   0.1113 │  57.321 │
├─────────────────┼───────┼────────────┼───────────┼──────────┼─────────┤
│ custom_eager    │ lstm  │     2.3041 │    0.1029 │   0.1064 │ 323.129 │
├─────────────────┼───────┼────────────┼───────────┼──────────┼─────────┤
│ custom_compiled │ lstm  │     2.3034 │    0.1079 │   0.1025 │  19.158 │
├─────────────────┼───────┼────────────┼───────────┼──────────┼─────────┤
│ jax             │ lstm  │     2.3154 │    0.0903 │   0.0967 │  19.718 │
├─────────────────┼───────┼────────────┼───────────┼──────────┼─────────┤
│ opacus          │ lstm  │     2.3019 │    0.1046 │   0.1143 │  94.678 │
└─────────────────┴───────┴────────────┴───────────┴──────────┴─────────┘